In [1]:
#import dependencies
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, f1_score
from sklearn.impute import SimpleImputer

import lightgbm as lgb

In [2]:
# Optional: hyperparameter tuning
try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Optuna not installed – skipping hyperparameter tuning.")

In [3]:
# Extinction correction
from extinction import fitzpatrick99

In [4]:
# Configuration
DATA_ROOT = r"C:\Users\sregi\Documents\ceartin_repo2\mallorn-astronomical-classification-challenge"
FILTERS = ['u', 'g', 'r', 'i', 'z', 'y']
FILTER_EFF_WL = {'u':3641, 'g':4704, 'r':6155, 'i':7504, 'z':8695, 'y':10056}

In [5]:
# Load log files
train_log = pd.read_csv(os.path.join(DATA_ROOT, "train_log.csv"))
test_log = pd.read_csv(os.path.join(DATA_ROOT, "test_log.csv"))

In [6]:
# Top features from correlation analysis (Cohen's d > 0.3) – per filter we will compute these
TOP_FEATURE_NAMES = [
    'beyond_1std', 'skew', 'kurtosis', 'stetson_k',
    'flux_mad', 'negative_flux_mean', 'zero_crossings',
    'half_light_duration'
]

In [7]:
# Utility functions
def correct_flux(row, ebv):
    eff_wl = FILTER_EFF_WL[row['Filter']]
    A_lambda = fitzpatrick99(np.array([eff_wl]), ebv * 3.1)[0]
    return row['Flux'] * 10**(A_lambda / 2.5)
    
def half_light_duration(time, flux, max_flux):
    if max_flux <= 0 or len(flux) < 2:
        return np.nan
    half = max_flux * 0.5
    above = flux > half
    if not np.any(above):
        return np.nan
    idx_first = np.where(above)[0][0]
    idx_last = np.where(above)[0][-1]
    return time[idx_last] - time[idx_first]

In [8]:
# Feature extraction for one object
def extract_features(obj_data, ebv):
    data = obj_data.copy()
    data['Flux_corr'] = data.apply(lambda r: correct_flux(r, ebv), axis=1)

    features = {}

    for f in FILTERS:
        fdata = data[data['Filter'] == f]
        prefix = f"{f}_"
        if len(fdata) == 0:
            features[prefix + 'mean'] = np.nan
            features[prefix + 'std'] = np.nan
            features[prefix + 'min'] = np.nan
            features[prefix + 'max'] = np.nan
            features[prefix + 'nobs'] = 0
            for feat in TOP_FEATURE_NAMES:
                features[prefix + feat] = np.nan
            continue

        flux = fdata['Flux_corr'].values
        flux_err = fdata['Flux_err'].values
        time = fdata['Time (MJD)'].values

        # Basic stats
        features[prefix + 'mean'] = np.mean(flux)
        features[prefix + 'std'] = np.std(flux)
        features[prefix + 'min'] = np.min(flux)
        features[prefix + 'max'] = np.max(flux)
        features[prefix + 'nobs'] = len(flux)

        # ---- Top engineered features ----
        mean = np.mean(flux)
        std = np.std(flux)
        features[prefix + 'beyond_1std'] = np.sum(np.abs(flux - mean) > std) / len(flux)

        features[prefix + 'skew'] = stats.skew(flux)
        features[prefix + 'kurtosis'] = stats.kurtosis(flux)

        features[prefix + 'flux_mad'] = np.median(np.abs(flux - np.median(flux)))

        neg_mask = flux < 0
        if np.any(neg_mask):
            features[prefix + 'negative_flux_mean'] = np.mean(flux[neg_mask])
        else:
            features[prefix + 'negative_flux_mean'] = 0.0

        if len(flux) > 1:
            sign_changes = np.sum(np.diff(np.sign(flux)) != 0)
            features[prefix + 'zero_crossings'] = sign_changes
        else:
            features[prefix + 'zero_crossings'] = 0

        if len(flux) >= 3:
            residuals = flux - np.median(flux)
            if np.std(residuals) > 0:
                features[prefix + 'stetson_k'] = np.sum(np.abs(residuals) / np.std(residuals)) / len(flux)
            else:
                features[prefix + 'stetson_k'] = 0
        else:
            features[prefix + 'stetson_k'] = 0

        features[prefix + 'half_light_duration'] = half_light_duration(
            time, flux, features[prefix + 'max']
        )

    features['total_obs'] = len(data)
    return features

In [9]:
# Process a split (train or test)
def process_split(split_name, log_subset, is_train=True):
    if is_train:
        lc_file = os.path.join(DATA_ROOT, split_name, "train_full_lightcurves.csv")
    else:
        lc_file = os.path.join(DATA_ROOT, split_name, "test_full_lightcurves.csv")

    lc_df = pd.read_csv(lc_file)
    merged = pd.merge(log_subset, lc_df, on="object_id", how="inner")

    records = []
    for obj_id, group in merged.groupby("object_id"):
        static = group.iloc[0][['Z', 'EBV']].to_dict()
        if is_train:
            static['target'] = group.iloc[0]['target']
        lc_feat = extract_features(group, static['EBV'])
        row = {**static, **lc_feat, 'object_id': obj_id}
        records.append(row)

    return pd.DataFrame(records)

In [10]:
# Load training data
print("Loading training data...")
splits = [f"split_{i:02d}" for i in range(1, 21)]
all_train = []
for split in tqdm(splits):
    log_part = train_log[train_log['split'] == split]
    if len(log_part) == 0:
        continue
    all_train.append(process_split(split, log_part, is_train=True))

train_df = pd.concat(all_train, ignore_index=True)
train_df = train_df.loc[:, ~train_df.columns.duplicated()]

feature_cols = [c for c in train_df.columns if c not in ['object_id', 'target']]
X = train_df[feature_cols]
y = train_df['target'].astype(int)

print(f"Training samples: {len(X)}")
print(f"Features: {len(feature_cols)}")
print("Target distribution:\n", y.value_counts())

# Impute missing values
imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X)
X = pd.DataFrame(X_imp, columns=feature_cols, index=X.index)

Loading training data...


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:43<00:00,  5.17s/it]

Training samples: 3043
Features: 81
Target distribution:
 target
0    2895
1     148
Name: count, dtype: int64


In [11]:
# Scale_pos_weight base
scale_pos_base = (y == 0).sum() / (y == 1).sum()
print(f"Base scale_pos_weight: {scale_pos_base:.2f}")

Base scale_pos_weight: 19.56


In [12]:
# Tune scale_pos_weight factor via cross‑validation (using LightGBM)
scale_factors = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
best_factor = 1.0
best_cv_score = 0
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Tuning scale_pos_weight factor...")
for factor in scale_factors:
    scores = []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w = scale_pos_base * factor
        model = lgb.LGBMClassifier(
            objective='binary',
            scale_pos_weight=w,
            n_estimators=300,
            learning_rate=0.01,
            num_leaves=63,
            subsample=0.6,
            colsample_bytree=0.6,
            random_state=42,
            verbose=-1
        )
        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_val)[:, 1]
        scores.append(average_precision_score(y_val, pred))
    mean_score = np.mean(scores)
    print(f"  factor={factor:.2f} -> PR-AUC = {mean_score:.4f}")
    if mean_score > best_cv_score:
        best_cv_score = mean_score
        best_factor = factor

scale_pos = scale_pos_base * best_factor
print(f"Selected scale_pos_weight = {scale_pos:.2f}")

Tuning scale_pos_weight factor...
  factor=0.50 -> PR-AUC = 0.3069
  factor=0.75 -> PR-AUC = 0.3097
  factor=1.00 -> PR-AUC = 0.3029
  factor=1.25 -> PR-AUC = 0.2974
  factor=1.50 -> PR-AUC = 0.3010
  factor=1.75 -> PR-AUC = 0.2992
  factor=2.00 -> PR-AUC = 0.2931
  factor=2.50 -> PR-AUC = 0.2842
  factor=3.00 -> PR-AUC = 0.2798
Selected scale_pos_weight = 14.67


In [13]:
# Hyperparameter tuning with Optuna (optional)
if OPTUNA_AVAILABLE:
    def objective(trial):
        params = {
            'num_leaves': trial.suggest_int('num_leaves', 15, 255),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        }

        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        cv_scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = lgb.LGBMClassifier(
                objective='binary',
                scale_pos_weight=scale_pos,
                n_estimators=500,
                random_state=42,
                verbose=-1,
                **params
            )
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='aucpr',
                      callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
            y_pred = model.predict_proba(X_val)[:, 1]
            cv_scores.append(average_precision_score(y_val, y_pred))
        return np.mean(cv_scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    best_params = study.best_params
    print("Best LightGBM params:", best_params)
else:
    best_params = {
        'num_leaves': 31,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_samples': 20,
        'reg_alpha': 1e-8,
        'reg_lambda': 1e-8
    }

[I 2026-03-15 18:53:23,510] A new study created in memory with name: no-name-c38de275-924c-4055-9ade-d61fc906c6ac


  0%|          | 0/30 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[179]	valid_0's binary_logloss: 0.155102
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[210]	valid_0's binary_logloss: 0.131736
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[273]	valid_0's binary_logloss: 0.137742
[I 2026-03-15 18:53:28,042] Trial 0 finished with value: 0.3221386447965596 and parameters: {'num_leaves': 154, 'learning_rate': 0.01914852952427387, 'subsample': 0.8889933186860517, 'colsample_bytree': 0.9562329548273764, 'min_child_samples': 46, 'reg_alpha': 0.004110244547988566, 'reg_lambda': 0.00014863786244303782}. Best is trial 0 with value: 0.3221386447965596.
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[21]	valid_0's binary_logloss: 0.173951
Training until validation scores don't improve for 50 rounds
Early stopping, best i

In [14]:
# Generate out‑of‑fold predictions for LightGBM (to tune threshold)
print("\nGenerating OOF predictions...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))

for train_idx, val_idx in tqdm(skf.split(X, y), total=5):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        objective='binary',
        scale_pos_weight=scale_pos,
        n_estimators=500,
        random_state=42,
        verbose=-1,
        **best_params
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='aucpr',
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

# Find best threshold on OOF (finer grid: 0.01 steps)
thresholds = np.arange(0.1, 0.91, 0.01)
best_thresh = 0.5
best_f1 = 0
for t in thresholds:
    y_pred_bin = (oof_preds >= t).astype(int)
    f1 = f1_score(y, y_pred_bin)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"Best threshold on OOF: {best_thresh:.2f} (F1 = {best_f1:.4f})")



Generating OOF predictions...


  0%|                                                                                            | 0/5 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds


 20%|████████████████▊                                                                   | 1/5 [00:00<00:03,  1.28it/s]

Early stopping, best iteration is:
[115]	valid_0's binary_logloss: 0.15234
Training until validation scores don't improve for 50 rounds


...

 80%|███████████████████████████████████████████████████████████████████▏                | 4/5 [00:03<00:00,  1.05it/s]

Early stopping, best iteration is:
[158]	valid_0's binary_logloss: 0.113121
Training until validation scores don't improve for 50 rounds


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.09it/s]

Early stopping, best iteration is:
[127]	valid_0's binary_logloss: 0.138894


Best threshold on OOF: 0.22 (F1 = 0.4341)


In [15]:
# Retrain final LightGBM on full training data
print("Retraining final model on full data...")
final_model = lgb.LGBMClassifier(
    objective='binary',
    scale_pos_weight=scale_pos,
    n_estimators=500,
    random_state=42,
    verbose=-1,
    **best_params
)
final_model.fit(X, y)

Retraining final model on full data...


,boosting_type,'gbdt'
,num_leaves,43
,max_depth,-1
,learning_rate,0.04996533648297814
,n_estimators,500
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,57


In [16]:
# Process test data
print("Processing test data...")
all_test = []
for split in tqdm(splits):
    log_part = test_log[test_log['split'] == split]
    if len(log_part) == 0:
        continue
    all_test.append(process_split(split, log_part, is_train=False))

test_df = pd.concat(all_test, ignore_index=True)
test_df = test_df.loc[:, ~test_df.columns.duplicated()]

X_test = test_df[feature_cols].copy()
X_test_imp = imputer.transform(X_test)
X_test = pd.DataFrame(X_test_imp, columns=feature_cols, index=test_df.index)

# Predict
test_pred_proba = final_model.predict_proba(X_test)[:, 1]
test_pred_labels = (test_pred_proba >= best_thresh).astype(int)

Processing test data...


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [04:13<00:00, 12.70s/it]


In [17]:
# Save submission
submission = pd.DataFrame({
    'object_id': test_df['object_id'],
    'target': test_pred_labels
})
submission.to_csv('lgbm_enhanced_submission.csv', index=False)

print("Submission saved to lgbm_enhanced_submission.csv")
print("Predictions distribution:", np.bincount(test_pred_labels))

Submission saved to lgbm_enhanced_submission.csv
Predictions distribution: [6892  243]
